# Prodigy InfoTech - Task 1: Text Generation with Fine-Tuned GPT-2

In [2]:
import torch

print("PyTorch Version:", torch.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch Version: 2.11.0+cu128
Device: cuda
GPU: Tesla T4


In [4]:
!pip install -q transformers datasets accelerate evaluate

In [5]:
import os
import math
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
    set_seed
)

set_seed(42)

In [6]:
raw_data = [
    {
        "prompt": "What is artificial intelligence?",
        "response": "Artificial intelligence is a branch of computer science dedicated to creating systems capable of performing tasks that typically require human cognitive abilities, such as reasoning, learning, and problem solving."
    },
    {
        "prompt": "What is machine learning?",
        "response": "Machine learning is a subset of artificial intelligence that allows algorithms to automatically learn patterns from empirical data and make accurate predictions without explicit hardcoded rules."
    },
    {
        "prompt": "What is deep learning?",
        "response": "Deep learning is a specialized subfield of machine learning based on multi-layered artificial neural networks that can automatically extract hierarchical representations from massive datasets."
    },
    {
        "prompt": "What is supervised learning?",
        "response": "Supervised learning is a machine learning paradigm where models are trained on input-output pairs with ground-truth labels to map new inputs to correct target outputs."
    },
    {
        "prompt": "What is unsupervised learning?",
        "response": "Unsupervised learning is a learning paradigm where algorithms identify latent structures, clusters, or probability distributions within unlabeled datasets."
    },
    {
        "prompt": "What is reinforcement learning?",
        "response": "Reinforcement learning is a computational framework where an autonomous agent learns optimal policies through sequential trial-and-error interactions with an environment using reward signals."
    },
    {
        "prompt": "What is a transformer model?",
        "response": "A transformer model is a state-of-the-art neural architecture that leverages self-attention mechanisms to process input sequences in parallel, capturing long-range dependencies efficiently."
    },
    {
        "prompt": "What is GPT-2?",
        "response": "GPT-2 (Generative Pre-trained Transformer 2) is an autoregressive causal language model developed by OpenAI that predicts the next token in a sequence using stacked masked transformer decoder blocks."
    },
    {
        "prompt": "What is an embedding in NLP?",
        "response": "An embedding is a dense, continuous numerical vector representation of words, tokens, or documents in a geometric space where semantically similar items reside close to each other."
    },
    {
        "prompt": "What is natural language processing?",
        "response": "Natural language processing (NLP) is an interdisciplinary field combining linguistics and machine learning to enable computers to comprehend, interpret, and generate natural human language."
    },
    {
        "prompt": "What is overfitting and how do you prevent it?",
        "response": "Overfitting occurs when a model memorizes noise in training data, degrading test generalization. It is mitigated using regularization, dropout, data augmentation, and early stopping."
    },
    {
        "prompt": "What is backpropagation?",
        "response": "Backpropagation is an efficient algorithm for computing gradients of the loss function with respect to neural network weights using the chain rule of calculus for optimization."
    },
    {
        "prompt": "What is transfer learning?",
        "response": "Transfer learning is a machine learning technique where knowledge gained from pre-training on a large generic dataset is fine-tuned to solve a specific downstream task with higher efficiency."
    },
    {
        "prompt": "What is causal language modeling?",
        "response": "Causal language modeling is a unidirectional sequence modeling task where each token is predicted strictly based on preceding tokens, preventing the model from looking ahead into the future."
    },
    {
        "prompt": "What is tokenization?",
        "response": "Tokenization is the process of segmenting raw text strings into discrete atomic units known as tokens, such as subwords, words, or characters, mapped to integer indices in a vocabulary."
    },
    {
        "prompt": "What is temperature in text generation?",
        "response": "Temperature is a hyperparameter that scales logits before softmax. Lower temperature (<1.0) produces deterministic and focused text, while higher temperature (>1.0) boosts diversity and creativity."
    },
    {
        "prompt": "What is top-p or nucleus sampling?",
        "response": "Top-p sampling restricts token selection to the smallest candidate set whose cumulative probability reaches threshold p, dynamically adapting the pool based on prediction certainty."
    },
    {
        "prompt": "What is gradient descent?",
        "response": "Gradient descent is a first-order iterative optimization algorithm that updates model parameters in the opposite direction of the loss gradient to minimize objective error."
    },
    {
        "prompt": "What is cross-entropy loss?",
        "response": "Cross-entropy loss quantifies the discrepancy between predicted probability distributions and true categorical targets, commonly utilized in multi-class classification and language modeling."
    },
    {
        "prompt": "What is attention mechanism?",
        "response": "An attention mechanism computes dynamic relevance weights between queries, keys, and values, allowing neural networks to focus selectively on salient portions of input sequences."
    }
]

formatted_data = [{"text": f"<|startoftext|>Question: {item['prompt']}\nAnswer: {item['response']}<|endoftext|>"} for item in raw_data]

dataset = Dataset.from_list(formatted_data)
split_dataset = dataset.train_test_split(test_size=0.15, seed=42)
print(split_dataset)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 17
    })
    test: Dataset({
        features: ['text'],
        num_rows: 3
    })
})


In [7]:
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id

print("Tokenizer loaded:", tokenizer)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Tokenizer loaded: GPT2Tokenizer(name_or_path='gpt2', vocab_size=50257, model_max_length=1024, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
})


In [8]:
def tokenize_function(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,
        padding="max_length"
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = split_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
print(tokenized_dataset)

Map:   0%|          | 0/17 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 17
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 3
    })
})


In [9]:
base_model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
base_model.eval()

test_prompt = "<|startoftext|>Question: What is machine learning?\nAnswer:"
input_ids = tokenizer.encode(test_prompt, return_tensors="pt").to(device)

with torch.no_grad():
    output_ids = base_model.generate(
        input_ids,
        max_new_tokens=60,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

print("--- Pre-Trained GPT-2 Output ---")
print(tokenizer.decode(output_ids[0], skip_special_tokens=False))

del base_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

--- Pre-Trained GPT-2 Output ---
<|startoftext|>Question: What is machine learning?
Answer: Machine learning is the application of machine learning algorithms to natural language processing, e.g., human language processing. The goal of machine learning is to reduce the need for human-like interactions with information.
Machine learning algorithms are often referred to as neural nets, and are generally used to perform artificial intelligence


In [10]:
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

output_dir = "./gpt2_finetuned_model"

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=8,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=5,
    save_steps=20,
    save_total_limit=1,
    eval_strategy="epoch",
    report_to="none",
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
)

train_result = trainer.train()
print(f"Training Loss: {train_result.training_loss:.4f}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,4.297840,2.716747
2,2.744827,2.428431
3,2.220035,2.403581
4,1.766956,2.445817
5,1.583472,2.521749
6,1.496298,2.583300
7,1.344990,2.635418
8,1.199323,2.654248


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss: 1.9748


In [11]:
save_path = "./saved_gpt2_custom_model"
os.makedirs(save_path, exist_ok=True)

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print("Model saved to:", save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: ./saved_gpt2_custom_model


In [12]:
fine_tuned_model = AutoModelForCausalLM.from_pretrained(save_path).to(device)
fine_tuned_model.eval()

def generate_answer(prompt_text, strategy="sampling", max_tokens=60, temperature=0.7, top_p=0.9, top_k=50, num_beams=3):
    formatted_prompt = f"<|startoftext|>Question: {prompt_text}\nAnswer:"
    input_encoded = tokenizer(formatted_prompt, return_tensors="pt").to(device)

    generation_kwargs = {
        "input_ids": input_encoded["input_ids"],
        "attention_mask": input_encoded["attention_mask"],
        "max_new_tokens": max_tokens,
        "pad_token_id": tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
        "no_repeat_ngram_size": 2,
    }

    if strategy == "greedy":
        generation_kwargs["do_sample"] = False
    elif strategy == "beam_search":
        generation_kwargs["do_sample"] = False
        generation_kwargs["num_beams"] = num_beams
        generation_kwargs["early_stopping"] = True
    elif strategy == "sampling":
        generation_kwargs["do_sample"] = True
        generation_kwargs["temperature"] = temperature
        generation_kwargs["top_p"] = top_p
        generation_kwargs["top_k"] = top_k

    with torch.no_grad():
        output_tokens = fine_tuned_model.generate(**generation_kwargs)

    full_text = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
    if "Answer:" in full_text:
        return full_text.split("Answer:", 1)[1].strip()
    return full_text.strip()

sample_q = "What is deep learning?"
print("Question:", sample_q)
print("Greedy:", generate_answer(sample_q, strategy="greedy"))
print("Beam Search:", generate_answer(sample_q, strategy="beam_search"))
print("Sampling:", generate_answer(sample_q, strategy="sampling"))

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Question: What is deep learning?
Greedy: Deep learning is a machine learning paradigm where algorithms automatically learn patterns from large datasets, allowing algorithms to automatically extract optimal patterns without human intervention. Deep neural networks can automatically process large amounts of data without requiring human cognitive abilities, such as deep neural nets, to do so.
Question, What are deep networks
Beam Search: Deep learning is a machine learning paradigm where algorithms automatically learn patterns from large datasets, allowing algorithms to automatically extract large amounts of information from unstructured data sets. It is an efficient, scalable, and efficient way to efficiently map large-scale datasets to machine-readable targets.
What is Deep Learning
Sampling: Deep learning is a branch of artificial intelligence that is capable of performing complex tasks efficiently, efficiently switching between tasks, and maintaining invariant state-of-the-art state m

In [13]:
eval_results = trainer.evaluate()
eval_loss = eval_results["eval_loss"]
perplexity = math.exp(eval_loss)

print(f"Evaluation Loss: {eval_loss:.4f}")
print(f"Perplexity (PPL): {perplexity:.2f}")

Training Loss,Validation Loss,Epoch
1.199323,2.654248,8


Evaluation Loss: 2.6542
Perplexity (PPL): 14.21


In [14]:
test_queries = [
    "What is a machine learning?",
    "What is linear regression?",
    "where to use linear regression?"
]

for q in test_queries:
    print("Question:", q)
    generated_text = generate_answer(q, strategy="sampling", temperature=0.6)

    idx_question = generated_text.find("Question")
    idx_answer = generated_text.find("Answer:", 1)

    truncate_idx = -1
    if idx_question != -1 and (truncate_idx == -1 or idx_question < truncate_idx):
        truncate_idx = idx_question
    if idx_answer != -1 and (truncate_idx == -1 or idx_answer < truncate_idx):
        truncate_idx = idx_answer

    if truncate_idx != -1:
        processed_answer = generated_text[:truncate_idx].strip()
    else:
        processed_answer = generated_text.strip()

    print("Answer:", processed_answer)
    print()

Question: What is a machine learning?
Answer: A machine-learning model is an artificial intelligence system that learns patterns from data, typically through supervised training. It can learn patterns using regularization, gradient descent, and reinforcement learning, as well as machine translation.

Question: What is linear regression?
Answer: Linear regression is a multi-model supervised learning paradigm where model selection is based on prior probability distributions, where each model is trained on a subset of the input sequence, resulting in a new model with better predictive power.

Question: where to use linear regression?
Answer: Linear regression is a non-parametric hierarchical model where each node in the model is predicted from the beginning of the sequence, where the smallest node is the first predicted node, and the largest node the last predicted.



In [15]:
custom_question = "What is natural language processing?"
response = generate_answer(custom_question, strategy="sampling", temperature=0.7)
print("Prompt:", custom_question)
print("Response:", response)

Prompt: What is natural language processing?
Response: Natural language process (NNP) is a computational paradigm where knowledge gained from deep learning can automatically extract patterns from human language representations and extract them from model-based datasets. It is modeled on top-down learning with high-throughput training, allowing machine learning to automatically identify patterns and iterate on
